In [ ]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU  = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")
    
        # 1. Faster Uninstallation 
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", 
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow"], 
                       capture_output=True)
    
        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]
        
        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0", 
                "torchvision==0.23.0", 
                "torch_xla[tpu]==2.8.0",
                # 🩹 THE FIX: Unpinned numpy, pyarrow, and fsspec to allow Numpy 2.0+ compatibility
                "numpy", "pyarrow", "fsspec", 
                "datasets", "transformers", "huggingface_hub", "wandb", 
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0", 
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)
    
        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")
    
        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", 
                        "torch", "torchvision", "torchaudio"], 
                       capture_output=True)
    
        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]
        
        try:
            print("📥 Installing GPU/CUDA Stack...")
            
            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])
            
            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub", 
                "wandb", "pandas<3.0.0" 
            ])
    
            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")
    
        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

In [ ]:
!pkill -9 -f python

In [6]:
!python parallel_hardware_trainer.py

Rank 0 is online: cuda:0.
⏳ Loading Checkpoint Driver...
✅ training_state.json loaded successfully from JamesResearch1216/HELM-v1-Architecture
Successfully loaded model and optimizer from Step 600!
Resolving data files: 100%|██████████████████| 71/71 [00:00<00:00, 30939.80it/s]
Step 601 | Total Loss: 5.4786 | CE: 5.4763 | Aux: 0.0000 | Sparsity: 0.0022
Step 602 | Total Loss: 5.6613 | CE: 5.6591 | Aux: 0.0000 | Sparsity: 0.0023
^C
🔪 Initiating Termination Sequence...
Process Process-1:
Shutdown complete.
Traceback (most recent call last):
  File "/kaggle/working/parallel_hardware_trainer.py", line 1111, in <module>
    driver.launch(train_worker)
  File "/kaggle/working/parallel_hardware_trainer.py", line 333, in launch
    mp.spawn(worker_fn, args=(self.hw_config, self.data_config, self.ckpt_config), nprocs=world_size)
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/spawn.py", line 328, in spawn
    return start_processes(fn, args, nprocs, join, daemon, start_meth

In [ ]:
%%writefile parallel_hardware_trainer.py
import os
import sys
import importlib
import time
import json
import site
import torch
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, create_repo, HfApi
from dataclasses import dataclass, field # Added field here
import multiprocessing
from typing import Optional, List, Union, ClassVar, Dict, Any
import warnings
from huggingface_hub.utils import RepositoryNotFoundError, EntryNotFoundError


# Disable Progress Bars
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except ImportError:
    pass

# Ensure PJRT runtime gets selected, not XRT
for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
    os.environ.pop(key, None)
os.environ["PJRT_DEVICE"] = "TPU"
# Add the framework quarantine just in case!
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

# Prevent C++ thread deadlocks during 10B token streaming
os.environ["OMP_NUM_THREADS"] = "1"
import pyarrow as pa
pa.set_cpu_count(1)
pa.set_io_thread_count(1)

# Tell everything to stay away from the TPU except PyTorch
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["XLA_USE_BF16"] = "1"
os.environ["XLA_DOWNCAST_BF16"] = "1"

# Force Path Refresh
if 'site' in sys.modules:
    importlib.reload(site)

# ==================================================
# HardwareConfig
# Keeps track of which device to use  what device-dependent values to use
# Not Static (Changes State)
# ==================================================

@dataclass
class HardwareConfig:

    HARDWARE_PROFILES = {
        "v5e-8": {
            "ws": 8, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 8, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": False, "sl": 2048},
            2: {"mb": 1, "use_ckpt": False, "sl": 4096},
        },
        "v5e-1": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 16, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": False, "sl": 2048},
            2: {"mb": 1, "use_ckpt": False, "sl": 4096},
        },
        "v6e-1": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 32, "use_ckpt": False, "sl": 1024},
            1: {"mb": 16, "use_ckpt": False, "sl": 2048},
            2: {"mb": 4, "use_ckpt": False, "sl": 4096},
        },
        "t4*2": {
            "ws": 2, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 4, "use_ckpt": True, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "t4": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 4, "use_ckpt": True, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "g4": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 4, "use_ckpt": False, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "l4": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 4, "use_ckpt": False, "sl": 1024},
            1: {"mb": 2, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "p100": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 8, "use_ckpt": True, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "a100": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 16, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "h100": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 16, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "cpu": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 1, "use_ckpt": False, "sl": 1024},
            1: {"mb": 1, "use_ckpt": False, "sl": 2048},
            2: {"mb": 1, "use_ckpt": False, "sl": 4096},
        }
    }

    hardware_string: str = "v5e-8 tpu"
    # hardware_string: str = "t4*2 gpu"
    hf_token: str = ""


    # These will be overwritten when we step thru the curriculum, but just place_holders for now
    world_size: int = 1
    target_gbs: int = 128
    dtype: torch.dtype = torch.float16
    use_scaler: bool = True
    batch_size: int = 16
    grad_accum_steps: int = 1
    hardware_profile: Dict[Union[str, int], Any] = field(
        default_factory=lambda: HardwareConfig.HARDWARE_PROFILES["cpu"]
    )
    device_type: str = "cpu"
    

# ==================================================
# MLMDataConfig
# Sets dataset repo, curriculum levels, and MLM parameters
# Static (Remains the same)
# ==================================================

@dataclass
class MLMDataConfig:
    data_repo_id: str = "JamesResearch1216/HELM-Processed-Data-10B"
    curriculum: bool = True
    curriculum_subset_names: List[str] = field(
        default_factory=lambda: ["seq_1024", "seq_2048", "seq_4096"]
    )
    # curriculum_subset_lens: List[int] = field(
    #     default_factory=lambda: [1024, 2048, 4096]
    # )
    train_split: str = "train"
    validation_split: str = "validation" 
    tokenizer_name: str = "answerdotai/ModernBERT-base"
    mlm_probability: float = 0.3
    mlm_use_span_masking: bool = True
    mlm_span_length: int = 3


# ==================================================
# CheckpointConfig
# Sets Checkpoint related fields
# Static (Remains the same)
# ==================================================

@dataclass
class CheckpointConfig:
    model_repo_id: str = "JamesResearch1216/HELM-v1-Architecture" 
    # repo_ver_override: Optional[int] = None
    hf_token: str = ""

    # How to use interval_dict:
    #  - Let k_i be the ith key in interval_dict
    #  - Let v_i be the ith value in interval_dict
    #  - For step k_i to k_i+1, save a checkpoint every v_i steps
    #  - After the last k_i, save every v_i for the rest of the duration
    interval_dict: Dict[int, int] = field(
        default_factory=lambda: {0: 100, 1000: 200, 5000: 500}
    )
    # True: let step 0 = latest_step
    # False: let step 0 = 0
    start_from_global: bool = True

    # This must be updated by calling the resume_training_step
    latest_step: int = -1



class MLMDataStrategy:

    # Initialize (Different for each hardware)
    def __init__(self, rank = 0, world_size = 1, is_tpu = False, config: Optional[MLMDataConfig] = None, hf_token=None):
        self.rank = rank
        self.world_size = world_size
        self.is_tpu = is_tpu
        self.config = config
        self.hf_token = hf_token
    
    # Create get_mlm_data_loader function
    # Note: This only bascially works with the dataset created by prepare_data.py

    def get_mlm_data_loader(self, collate_fn = None, skip_rows = 0, batch_size = 1, curriculum_level = 0, is_train = True):

        # Set up Dataset Differently for curriculum
        if (self.config.curriculum):
            dataset = load_dataset(
                path = self.config.data_repo_id, 
                name = self.config.curriculum_subset_names[curriculum_level], 
                split = self.config.train_split if is_train else self.config.validation_split,
                streaming = True,
                token=self.hf_token
            )
        else:
            # Else just load normally
            dataset = load_dataset(
                path = self.config.data_repo_id, 
                name = self.config.dataset_subset, 
                split = self.config.train_split if is_train else self.config.validation_split,
                streaming = True,
                token=self.hf_token
            )

        # IF we're Parallel Processing, Shard the Data so that each device gets a different slice of data
        if self.world_size > 1 and is_train:
            dataset = dataset.shard(num_shards = self.world_size, index = self.rank)

        # Shuffle after you shard
        # Make sure you set a seed to ensure the I don't use the same data again
        dataset = dataset.shuffle(buffer_size = 10000, seed = 67)
       
        # Skip examples after you shuffle based on the specific seed:
        dataset = dataset.skip(skip_rows)

        # set workers = 0 for TPUs else actually the real number of CPU cores
        # this was in the 1.x_model_trainer series of scripts
        # workers = 0 if self.is_tpu else max(1, multiprocessing.cpu_count()-1)
        # Just set workers to 0. Don't wanna crash our CPU, right?

        data_loader = DataLoader(
            dataset, 
            batch_size = batch_size, 
            num_workers = 0, 
            drop_last = True, 
            pin_memory = False,  
            collate_fn = collate_fn
        )

        # Return data_loader
        return data_loader



class HardwareDriver:

    # Initialize Hardware
    def __init__(self, hw_config: HardwareConfig, data_config: MLMDataConfig, ckpt_config: CheckpointConfig):
        self.hw_config = hw_config
        self.data_config = data_config
        self.ckpt_config = ckpt_config
        # Call _parse_hardware here
        self.hw_config.hardware_profile = self._parse_hardware()
    

    # Parse hardware based on the curriculum level
    def _parse_hardware(self):
        # get hardware_string (with formatting)
        hardware_string = self.hw_config.hardware_string.lower().replace(" ", "")

        # Ensure that the hardware_string is valid
        profile = None
        for key in self.hw_config.HARDWARE_PROFILES:
            if (key in hardware_string):
                profile = self.hw_config.HARDWARE_PROFILES[key]
                break

        # Default to cpu if none were matching
        if not profile:
            profile = self.hw_config.HARDWARE_PROFILES["cpu"]
            warnings.warn("⚠️ hardware_string did not match any in HARDWARE_PROFILES. Using \"cpu\"", UserWarning)

        # Add attributes to config common to all curriculum levels
        #   - world_size
        #   - targt_gbs (global batch size)
        #   - dtype (data type)
        #   - use_scaler
        self.hw_config.world_size = profile["ws"]
        self.hw_config.target_gbs = profile["target"]
        self.hw_config.dtype = profile["dtype"]
        self.hw_config.use_scaler = profile["use_scaler"]

        # Get device type (save to hw_config)
        if "tpu" in hardware_string:
            self.hw_config.device_type = "tpu"
        elif any(x in hardware_string for x in ["gpu", "cuda", "a100", "p100", "h100", "t4", "l4"]):
            self.hw_config.device_type = "cuda"
        else:
            self.hw_config.device_type = "cpu"
        
        # Return the profile to use in train_worker
        return profile
    
    # Launch function: Spawn all the workers and make them run the worker_function
    def launch(self, worker_fn):

        # Define these for convience
        world_size = self.hw_config.world_size
        device = self.hw_config.device_type
        
        # If parallel processing
        if world_size > 1:
            if device == "tpu":
                import torch_xla.distributed.xla_multiprocessing as xmp
                xmp.spawn(worker_fn, args=(self.hw_config, self.data_config, self.ckpt_config), start_method='spawn')
            elif device == "cuda":
                import random
                import torch.multiprocessing as mp
                
                # Set up Multi-GPU network
                os.environ['MASTER_ADDR'] = 'localhost'
                os.environ['MASTER_PORT'] = str(random.randint(10001, 19999))


                mp.spawn(worker_fn, args=(self.hw_config, self.data_config, self.ckpt_config), nprocs=world_size)
        else:
            # Single Device Execution (Rank 0)
            worker_fn(0, self.hw_config, self.data_config, self.ckpt_config)



class CheckpointDriver:

    # Initialize Checkpoint Driver
    def __init__(self, hw_config: HardwareConfig, ckpt_config: CheckpointConfig, rank: int, world_size: int):
        self.hw_config = hw_config
        self.checkpoint_config = ckpt_config
        self.rank = rank
        self.world_size = world_size
        self.api = HfApi(token=self.checkpoint_config.hf_token)
        self.actual_resume_step = None

        # Just print to ensure shit is moving
        if rank == 0:
            print("⏳ Loading Checkpoint Driver...")

        self.training_state = self.get_training_state_from_hub()


    # Smart Barrier (Rendezvous) to prevent data races & ensure all devices make it to certain step
    def _smart_barrier(self, name="barrier"):
        if self.hw_config.world_size <= 1:
            return  # No synchronization needed for single device
            
        if self.hw_config.device_type == "tpu":
            import torch_xla.core.xla_model as xm
            xm.rendezvous(name)
        elif self.hw_config.device_type == "cuda":
            import torch.distributed as dist
            if dist.is_initialized():
                dist.barrier()    


    # Initialize new checkpoint dictionary
    def _init_new_training_state(self):
        training_state_dict = {  
            "checkpoints": {}
        }
        return training_state_dict
    
    
    # Ensure that if checkpoints are deleted but appear 
    def _deletion_status_updates(self, training_state):
        
        # Try to take the repo file's file paths
        repo_files = None
        try:
            repo_files = list(self.api.list_repo_files(repo_id = self.checkpoint_config.model_repo_id))
        except RepositoryNotFoundError:
            raise RepositoryNotFoundError(f"❌ Repo: \"{self.checkpoint_config.model_repo_id}\" was not found when trying to update deletion status")
            

        # Loop Through every checkpoint and switch the status if necessary
        for ckpt_vals in training_state["checkpoints"].values():
            if ckpt_vals["file"] == "" or not ckpt_vals["file"] in repo_files:
                ckpt_vals["status"] = "deleted"
                ckpt_vals["file"] = ""

                
        
        # Return training_state
        return training_state


    # Get training_state.json from the HF repo
    def get_training_state_from_hub(self, filename = "training_state.json"):

        # Let only rank 0 to run this (prevent mass API calls)
        if self.world_size > 1:
            self._smart_barrier("state_fetch_start")

        # Let rank == 0 load the .json
        if self.rank == 0:
            # Attempt to pull the training_state.json from hub
            try:
                # Try Downlaoding
                path = hf_hub_download(repo_id=self.checkpoint_config.model_repo_id, filename = filename, repo_type = "model")
                with open(path, "r") as f:
                    training_state = json.load(f)

                # Successfully loaded
                print(f"✅ {filename} loaded successfully from {self.checkpoint_config.model_repo_id}")
                training_state = self._deletion_status_updates(training_state)
            
            # If the repo doesn't exist, make the repo
            except RepositoryNotFoundError:
                # Print Error Statements
                print(f"⚠️ Repo: \"{self.checkpoint_config.model_repo_id}\" was not found")
                print(f"🏗️ Creating Repo: {self.checkpoint_config.model_repo_id}")

                # Create Repo
                create_repo(
                    repo_id = self.checkpoint_config.model_repo_id,
                    token = self.checkpoint_config.hf_token,
                    repo_type = "model",
                    private = False,
                    exist_ok = False
                )

                # Make new training_state dict
                training_state = self._init_new_training_state()
            
            # The repo exists, but it's empty or doesn't have the state file yet
            except EntryNotFoundError:
                # Print info
                print(f"⚠️ {self.checkpoint_config.model_repo_id} exists, but no {filename} found. Starting fresh.")
                training_state = self._init_new_training_state()
            
            # Unknown Error
            except Exception as e:
                # Catch-all for network timeouts, corrupted JSON, etc.
                print(f"❌ An unexpected error occurred: {e}. Starting from token zero.")
                training_state = self._init_new_training_state()
            
            # Dump the training_state from rank 0 into .json
            with open("local_training_state.json", "w") as f:
                json.dump(training_state, f)
        
        # Once rank 0 finishes, end the barrier
        if self.world_size > 1:
            self._smart_barrier("state_fetch_end")
        
        # Then every rank (including) loads the dict from "local_training_state.json"
        with open("local_training_state.json", "r") as f:
            final_state_dict = json.load(f)

        # Wait for EVERY rank to finish reading the file
        if self.world_size > 1:
            self._smart_barrier("state_read_complete")

        # Then Delete
        if self.rank == 0:
            import os
            if os.path.exists("local_training_state.json"):
                os.remove("local_training_state.json")

        # All Return the same dict
        return final_state_dict

    # Check to see if a checkpoint should be uploaded
    def check_upload_condition(self, curr_global_step):
        # Subtract offset if start_from_global (it treated step 0 = last checkpoint's step value)
        if not self.checkpoint_config.start_from_global:
            curr_global_step -= self.actual_resume_step
        if curr_global_step <=0:
            return False
        
        # Save which interval we will use to calculate if we need to upload
        active_interval = None

        # Iterate through the sorted keys
        for threshold in sorted(self.checkpoint_config.interval_dict.keys()):
            # If our curr_global_step is bigger than threshold, save it's value
            if curr_global_step >= threshold:
                active_interval = self.checkpoint_config.interval_dict[threshold]
            else:
                # else we break since we haven't to this threshold yet
                break

        # If dictionary was empty (no checkpointing)
        if active_interval is None:
            return False
        
        # return whether the current step is a perfect multiple of the active_interval
        return (curr_global_step % active_interval == 0)
    
    # Resume Training: resume from the correct checkpoint
    # Pass the model and optimizer by reference to be initialized
    # Returns:
    # Checkpoint Entry Dictionary Snapshot if available
    # Dicionary with a bunch of 0s if all checkpoints were deleted or starting fresh
    # None if other errors (fast failing)
    def resume_training(self, model, optimizer):
        # Lazy Load torch to get correct version
        import torch
        
        # Keep track of all the valid steps
        valid_steps = []
        for step, data in self.training_state["checkpoints"].items():
            if data["status"] != "deleted":
                valid_steps.append(int(step))
        
        # Print messege and return 0 if its brand new
        if not valid_steps:
            if self.rank == 0:
                print("According to the training_state, every single checkpoint is invalid or deleted. Starting from ground 0")
            # return 0,0
            self.actual_resume_step = 0
            return {
                "curriculum_level": 0,
                "rows_processed_at_curr_level": 0,
                "total_tokens_processed_global": 0,
                "total_rows_processed_global": 0,  
            }, 0
        
        # Get Actual valid resume step (e.g. I deleted the most recent version but it still says otherwise)
        actual_resume_step = max(valid_steps)
        self.actual_resume_step = actual_resume_step
        ckpt_entry = self.training_state["checkpoints"][str(actual_resume_step)]
        filename = ckpt_entry["file"]

        # Barrier
        if self.world_size > 1:
            self._smart_barrier("weight_download_start")
        
        # Only let rank 0 start downloading (the others will download from the runtime local disk):
        if self.rank == 0:
            print(f"Downloading {filename} from Hub...")
            try:
                hf_hub_download(
                    repo_id = self.checkpoint_config.model_repo_id,
                    filename = filename,
                    repo_type = "model",
                    token = self.checkpoint_config.hf_token,
                    local_dir = "."
                )
            except Exception as e:
                raise RuntimeError(f"Critical HF Download Failure for {filename}: {e}")

        # Barrier
        if self.world_size > 1:
            self._smart_barrier("weight_download_end")
        
        # Now that the model has been downloaded onto the runtime local disk, let each device download it
        # All ranks load the weights from the local file
        try:

            # Load the checkpoint
            pt_path = os.path.join(".", filename)
            ckpt = torch.load(pt_path, map_location='cpu', weights_only=False)
            
            # Get the model state
            model_state = ckpt['model_state']

            # GPUs and TPUs might add module. or not have it at all
            # Add or subtract this to maintain hardware compatibility
            new_state_dict = {}
            for k, v in model_state.items():
                if k.startswith('module.') and not hasattr(model, 'module'):
                    new_state_dict[k[7:]] = v
                elif not k.startswith('module.') and hasattr(model, 'module'):
                    new_state_dict[f'module.{k}'] = v
                else:
                    new_state_dict[k] = v
            
            # Load the model into dictionary
            model.load_state_dict(new_state_dict, strict=False)
            
            # Load the optmizer
            if optimizer and 'optimizer_state' in ckpt:
                optimizer.load_state_dict(ckpt['optimizer_state'])
            
            # print success
            if self.rank == 0:
                print(f"Successfully loaded model and optimizer from Step {actual_resume_step}!")
            
            # Just return the ckpt_entry; Extract the values later
            return ckpt_entry, actual_resume_step
            
        except Exception as e:
            raise RuntimeError(f"Critical Weight Loading Failure: {e}")

    # Saves the model to a .pt file
    # Makes checkpoint entry for training_state
    def save_checkpoint(self, model, optimizer, global_step: int, hardware_string: str, metrics: dict, is_tpu: bool, curriculum_level: int, total_tokens_processed_global: int, total_rows_processed_global: int, rows_processed_at_curr_level: int):
        # Lazy Load Torch
        import torch 
        
        # Save filename
        step_str = str(global_step)
        filename = f"checkpoint-{step_str}.pt"

        # If more than 1 worker, start barrier
        if self.world_size > 1:
            self._smart_barrier("save_start")

        if self.rank == 0:
            print(f"Saving model weights to {filename}...")
        
        # Ensure to use module or not to ensure compatibility
        save_dict = {
            "model_state": model.module.state_dict() if hasattr(model, "module") else model.state_dict(),
            "optimizer_state": optimizer.state_dict() 
        }

        # Save using TPU or GPU .save()
        if is_tpu:
            import torch_xla.core.xla_model as xm
            xm.save(save_dict, filename)
        else:
            torch.save(save_dict, filename)

        # If more than 1 worker, end barrier
        if self.world_size > 1:
            self._smart_barrier("save_weights_end")

        # Update training_state (add checkpoint entry + update metadata)
        # Only let rank 0 change the state of the UPLOAD_REQUEST.json to ping the sidecar
        if self.rank == 0:

            # Ensure that the all latest tags get removed
            for step, data in self.training_state['checkpoints'].items():
                if data["status"] == "latest":
                    data["status"] = "history"
            
            # Create new checkpoint entry
            self.training_state["checkpoints"][step_str] = {
                "status": "latest",
                "file": filename,
                "hardware": hardware_string,
                "curriculum_level": curriculum_level, 
                "rows_processed_at_curr_level": rows_processed_at_curr_level,
                "total_rows_processed_global": total_rows_processed_global,
                "total_tokens_processed_global": total_tokens_processed_global,
                "metrics": metrics, 
            }

            # Dump new training_state into training_state.json
            with open("training_state.json", "w") as f:
                json.dump(self.training_state, f, indent = 4)

            # Ping sidecar by updating UPLOAD_REQUEST.json
            request_data = {
                "file_to_upload": filename, 
                "step": global_step,
                "training_state_snapshot": self.training_state
            }

            with open(f"UPLOAD_REQUEST_{global_step}.json.tmp", "w") as f:
                json.dump(request_data, f)
            os.rename(f"UPLOAD_REQUEST_{global_step}.json.tmp", f"UPLOAD_REQUEST_{global_step}.json")
            
            # Print some bs idk lol
            print(f"Saved weights to local disk + updated training_state.json. Pinging Sidecar for Step{global_step}")

        # If more than 1 worker make sure other ranks wait for rank 0
        if self.world_size > 1:
            self._smart_barrier("save_training_state_end")   



# Function that all devices will run (ran from the launch function right above)
def train_worker(rank, hw_config, data_config, ckpt_config):

    # Lazy Load
    import sys
    import traceback
    import os # Add os
    
    os.environ["HF_TOKEN"] = hw_config.hf_token
    
    
    try:
        # Lazy Load
        import datasets
        datasets.config.TF_AVAILABLE = False
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from transformers import AutoTokenizer
        import SpanMLMCollator
        from model import HELMConfig, HELMForMaskedLM
        

        # Default for Data Collator
        is_tpu = False

        # Load correct packages and get device
        if hw_config.device_type == "tpu":
            # Lazy Load even more for TPU
            import torch_xla.core.xla_model as xm
            import torch_xla.distributed.parallel_loader as pl
            import torch_xla.runtime as xr

            device = xm.xla_device()
            is_tpu = True

            # get real world size just in case
            real_world_size = xr.world_size()
            if hw_config.world_size != real_world_size:
                if rank == 0:
                    print(f"⚠️ CONFIG MISMATCH: Adjusting world size to {real_world_size}")
                hw_config.world_size = real_world_size
        
        elif hw_config.device_type == "cuda":
            # Set cuda device to torch
            torch.cuda.set_device(rank)
            device = torch.device(f"cuda:{rank}")
            # Initialize Distributed comm framework
            # acts like a rendezvous
            # NVIDIA Collective Communications Library (nccl)
            if hw_config.world_size > 1:
                import torch.distributed as dist
                dist.init_process_group("nccl", rank=rank, world_size=hw_config.world_size)
       
        else:
            # Default to CPU just in case
            device = torch.device("cpu")
        

        if rank == 0:
            print(f"Rank 0 is online: {device}.")
        
        # Define Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            data_config.tokenizer_name, token=hw_config.hf_token
        )

        # Define MLMDataStrategy
        data_strat = MLMDataStrategy(
            rank = rank, world_size = hw_config.world_size, is_tpu = is_tpu,config = data_config, hf_token=hw_config.hf_token
        ) 
  
        # Initialize Config
        helm_config = HELMConfig(
            vocab_size=len(tokenizer),
            pad_token_id=tokenizer.pad_token_id,
        )

        # Create model and attach to device
        model = HELMForMaskedLM(helm_config).to(device)

        # Require DDP to Wrap the model if using cuda
        if hw_config.device_type == "cuda" and hw_config.world_size > 1:
            from torch.nn.parallel import DistributedDataParallel as DDP
            # model = DDP(model, device_ids=[rank], find_unused_parameters=True)
            # There shouldn't be extra args
            model = DDP(model, device_ids=[rank])
    
        # Define Optimizer
        optimizer = optim.AdamW(model.parameters(), lr = helm_config.lr)

        # Zero the gradient
        optimizer.zero_grad()

        # Define CE Loss
        loss_fct = nn.CrossEntropyLoss()
        
        # Set data type that will be used
        dtype = hw_config.dtype

        # Allow Scaler
        use_scaler = hw_config.use_scaler
        scaler = torch.amp.GradScaler('cuda') if hw_config.device_type == "cuda" and use_scaler else None

        # ========== CHECKPOINT TECHNOLOGICA ==========

        # Define Checkpoint Driver
        checkpoint_driver = CheckpointDriver(
            hw_config = hw_config, ckpt_config = ckpt_config, rank = rank, world_size = hw_config.world_size
        )

        # Loading the model/optimizer returns the most recent, valid / undeleted checkpoint
        ckpt_snapshot, actual_resume_step = checkpoint_driver.resume_training(model, optimizer)

        # Extract the values from the ckpt_snapshot
        start_curr_level = ckpt_snapshot["curriculum_level"]
        rows_processed_at_curr_level = ckpt_snapshot["rows_processed_at_curr_level"]
        total_rows_processed_global = ckpt_snapshot["total_rows_processed_global"]
        total_tokens_processed_global = ckpt_snapshot["total_tokens_processed_global"]

        # Set the global step to where we left off from the previous checkpoint
        global_step = actual_resume_step

        # If starting fresh initialize weights
        if actual_resume_step == 0:
            model.apply(model._init_weights)
        
        
        # Curriculum Outer Loop (starting from the current curriculum):
        for level in range(start_curr_level,len(data_config.curriculum_subset_names)):

            # Set Model to Training Mode
            model.train()

            # Get Profile Level
            level_profile = hw_config.hardware_profile[level] 

            # Get micro batch size (mb) and use gradient checkpointing (use_ckpt)
            hw_config.batch_size = level_profile["mb"]
            
            # CRITICAL: Unwrap the model first to handle DDP (GPUs) vs Raw (TPUs)
            unwrap_model = model.module if hasattr(model, "module") else model
            # Change the Model Configs using the safely unwrapped model
            unwrap_model.config.use_ckpt = level_profile["use_ckpt"]
            unwrap_model.model.use_ckpt = level_profile["use_ckpt"] # HELMModel caches this
            
            # Save seq_len somewhere just in case if we need to use it
            seq_len = level_profile["sl"]

            # Calculate grad_accum_steps
            hw_config.grad_accum_steps = max(1, hw_config.target_gbs // (hw_config.batch_size * hw_config.world_size))

            # Define the Collator
            collator = SpanMLMCollator.SpanMLMCollator(
                config = data_config, tokenizer = tokenizer
            )
           
            # Define train_dataloader
            train_loader = data_strat.get_mlm_data_loader(
                collate_fn = collator, skip_rows = rows_processed_at_curr_level, batch_size = hw_config.batch_size, curriculum_level = level, is_train = True
            )

            # Define validation_dataloader
            validation_loader = data_strat.get_mlm_data_loader(
                collate_fn = collator, batch_size = hw_config.batch_size, curriculum_level = level, is_train = False
            )

            # if TPU is being used, apply the ParallelLoader().per_device_loader()
            if is_tpu:
                train_loader = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)
                validation_loader = pl.ParallelLoader(validation_loader, [device]).per_device_loader(device)
                
            # ========== TRAINING LOOP ==========
            # Loop through each batch
            for step, batch in enumerate(train_loader):
                
                # Get Batch's input ids, labels, and attn_mask (we don't have one but just in case) and attach it to device
                input_ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)
                attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)

                # GPUs require Autocast for Mixed Precision. TPUs handle it natively via Env Variables.
                if hw_config.device_type == "cuda":
                    with torch.autocast(device_type="cuda", dtype=dtype):
                        logits, aux_loss, sparsity_loss = model(input_ids=input_ids, attention_mask=attention_mask, current_step=global_step)
                        # logits: [mb, seq_len, vocab_size] -> [mb*seq_len, vocab_size]
                        # labels: [mb, seq_len], [mb*seq_len]
                        ce_loss = loss_fct(logits.view(-1, helm_config.vocab_size), labels.view(-1))
                        total_loss = (ce_loss + aux_loss + sparsity_loss) / hw_config.grad_accum_steps
                else:
                    logits, aux_loss, sparsity_loss = model(input_ids=input_ids, attention_mask=attention_mask, current_step=global_step)
                    # logits: [mb, seq_len, vocab_size] -> [mb*seq_len, vocab_size]
                    # labels: [mb, seq_len], [mb*seq_len]                    
                    ce_loss = loss_fct(logits.view(-1, helm_config.vocab_size), labels.view(-1))
                    total_loss = (ce_loss + aux_loss + sparsity_loss) / hw_config.grad_accum_steps
                
                if scaler is not None:
                    scaler.scale(total_loss).backward()
                else:
                    total_loss.backward()

                # Once Gradient has been accumulated, step the model and the optimizer
                if (step + 1) % hw_config.grad_accum_steps == 0:
                    if is_tpu:
                        xm.optimizer_step(optimizer)
                        xm.mark_step()
                    elif scaler is not None:
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()

                    # Zero the gradient
                    optimizer.zero_grad()
                    global_step += 1

                    # Calculating the values for the save_checkpoint
                    # Should just be the target gbs, but just in case
                    rows_this_step = hw_config.batch_size * hw_config.grad_accum_steps * hw_config.world_size
                    tokens_this_step = rows_this_step * seq_len

                    rows_processed_at_curr_level +=  rows_this_step
                    total_rows_processed_global += rows_this_step
                    total_tokens_processed_global += tokens_this_step
                    
                    # Use 1 device (rank = 0) to calculate the real loss
                    if rank == 0:
                        print(f"Step {global_step} | Total Loss: {total_loss.item()*hw_config.grad_accum_steps:.4f} | CE: {ce_loss.item():.4f} | Aux: {aux_loss if isinstance(aux_loss, float) else aux_loss.item():.4f} | Sparsity: {sparsity_loss if isinstance(sparsity_loss, float) else sparsity_loss.item():.4f}")   

                    # Save the model if the time is right (based on interval_dict from CheckpoingConfig)
                    if checkpoint_driver.check_upload_condition(global_step):
                        checkpoint_driver.save_checkpoint(
                            model = model, 
                            optimizer = optimizer, 
                            global_step = global_step, 
                            hardware_string = hw_config.hardware_string,
                            metrics = {
                                "Total Loss" :  round(total_loss.item()*hw_config.grad_accum_steps,5),
                                "CE Loss" : round(ce_loss.item(),5),
                                "AUX Loss" : round(aux_loss.item(),5),
                                "Sparsity" : round(sparsity_loss.item(),5)

                            }, 
                            is_tpu = is_tpu, 
                            curriculum_level = level,
                            total_tokens_processed_global = total_tokens_processed_global,
                            total_rows_processed_global = total_rows_processed_global,
                            rows_processed_at_curr_level = rows_processed_at_curr_level
                        )
            
            # zero the gradient again just in case
            optimizer.zero_grad()



            # ========== VALIDATION LOOP ==========

            # Put model into eval mode
            model.eval()

            # Initialize accumulators
            total_val_loss = 0.0
            total_ce_loss = 0.0
            total_aux_loss = 0.0
            total_sparsity_loss = 0.0
            step_count = 0

            # Loop through the validation_loader
            for step, batch in enumerate(validation_loader):
                step_count += 1
                
                # Get Batch's input ids, labels, and attn_mask and attach it to device
                input_ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)
                attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)

                # Use no_grad to prevent OOM during validation
                with torch.no_grad():
                    # GPUs require Autocast for Mixed Precision. TPUs handle it natively via Env Variables.
                    if hw_config.device_type == "cuda":
                        with torch.autocast(device_type="cuda", dtype=dtype):
                            logits, aux_loss, sparsity_loss = model(input_ids=input_ids, attention_mask=attention_mask)
                            ce_loss = loss_fct(logits.view(-1, helm_config.vocab_size), labels.view(-1))
                            val_loss = ce_loss + aux_loss + sparsity_loss
                    else:
                        logits, aux_loss, sparsity_loss = model(input_ids=input_ids, attention_mask=attention_mask)
                        ce_loss = loss_fct(logits.view(-1, helm_config.vocab_size), labels.view(-1))
                        val_loss = ce_loss + aux_loss + sparsity_loss
                
                # Get aux and sparsity values
                current_aux = aux_loss if isinstance(aux_loss, float) else aux_loss.item()
                current_sparsity = sparsity_loss if isinstance(sparsity_loss, float) else sparsity_loss.item()
                
                # Add Loss Values
                total_val_loss += val_loss.item()
                total_ce_loss += ce_loss.item()
                total_aux_loss += current_aux
                total_sparsity_loss += current_sparsity

                # Print Validation every so ofter
                if rank == 0 and step_count % 100 == 0:
                    print(f"Validation Step {step_count} | Batch Loss: {val_loss.item():.4f}")

            # Calculate and print the final averages once the loop naturally finishes
            if step_count > 0:
                avg_val_loss = total_val_loss / step_count
                avg_ce_loss = total_ce_loss / step_count
                avg_aux_loss = total_aux_loss / step_count
                avg_sparsity_loss = total_sparsity_loss / step_count
                
                if rank == 0:
                    print(f"Average Total Loss: {avg_val_loss:.4f} | Avg CE: {avg_ce_loss:.4f} | Avg Aux: {avg_aux_loss:.4f} | Avg Sparsity: {avg_sparsity_loss:.4f}\n")

        # Destroy once all of these johns are done
        if hw_config.device_type == "cuda" and hw_config.world_size > 1:
            dist.destroy_process_group()
    
    except Exception as e:
        print(f"\n❌ FATAL WORKER ERROR ON RANK {rank}:")
        traceback.print_exc()
        return



def sidecar_uploader_loop(hf_token, repo_id):
    # LAZY LOAD
    import os
    import json
    import time
    from datetime import datetime, timezone
    from huggingface_hub import HfApi

    # Get HF API Token to upload
    api = HfApi(token=hf_token)

    # Forever Loop to constantly check
    while True:
        
        # Check to see if any valid upload requests exist & take the step size
        upload_requests = []
        for file in os.listdir("."):
            if file.startswith("UPLOAD_REQUEST_") and file.endswith(".json"):
                upload_requests.append(int(file.replace("UPLOAD_REQUEST_", "").replace(".json", "")))
        
        # Sort the list and take the first request
        if upload_requests:
            
            # Get the next upload_request and process that first
            next_upload = sorted(upload_requests)[0]
            upload_request_filename = f"UPLOAD_REQUEST_{next_upload}.json"

            # Try to upload the model and training_state.json to HF HUB
            try:
                
                # Open the UPLOADER_REQUEST.json
                with open(upload_request_filename, "r") as f:
                    UPLOAD_REQUEST = json.load(f)
                
                # Get filename and the step
                model_filename = UPLOAD_REQUEST["file_to_upload"]
                step = UPLOAD_REQUEST["step"]
                training_state_snapshot = UPLOAD_REQUEST["training_state_snapshot"]

                # Print Messeage
                print(f"⏳ Attempting to upload {model_filename} to {repo_id}")

                # Upload the model first (most unstable action to do before uplaoding the .json)
                if os.path.exists(model_filename):
                    api.upload_file(
                        path_or_fileobj=model_filename,
                        path_in_repo=model_filename,
                        repo_id=repo_id,
                        repo_type="model"
                    )
                else:
                    print(f"❌ Failed to Upload. {upload_request_filename} was pinged, but {model_filename} does not exist")

                # Dump the snapshot's training state to a temporary .json
                with open(f"uploading_training_state.json", "w") as f:
                    json.dump(training_state_snapshot, f)
                
                # Upload the training_state.json
                api.upload_file(
                    path_or_fileobj="uploading_training_state.json",
                    path_in_repo="training_state.json",
                    repo_id=repo_id,
                    repo_type="model"
                )
                

                # Squash History to ensure that the repo doesn't hold onto the archives
                api.super_squash_history(repo_id=repo_id)

                # Remove the current checkpoint and UPLOAD_REQUEST_XXXXXX.json, and uploading_training_state.json
                os.remove(model_filename)
                os.remove(upload_request_filename)
                os.remove("uploading_training_state.json")

                print(f"✅ Successfully uploaded {model_filename} @ step {step} to {repo_id}")

            except Exception as e:
                print(f"❌ Failed to upload to HF: {e}")
                print("Trying again in 5 seconds...")

        # Pause 5 seconds before rechecking if UPLOAD_REQUEST.json exists
        time.sleep(5)


if __name__ == "__main__":
    # Ensure environment is primed for TPU PJRT
    for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
        os.environ.pop(key, None)
    os.environ["PJRT_DEVICE"] = "TPU"

    # Initialize all configs
    HW_CFG = HardwareConfig()
    DATA_CFG = MLMDataConfig()
    CKPT_CFG = CheckpointConfig()

    # Loading Sidecar via isolated CPU thread to upload
    uploader_process = multiprocessing.Process(
        target = sidecar_uploader_loop,
        args = (CKPT_CFG.hf_token, CKPT_CFG.model_repo_id)
    )
    uploader_process.start()
    
    # Prepare Hardware Driver
    driver = HardwareDriver(HW_CFG, DATA_CFG, CKPT_CFG)

    # Try to launch training process
    try:
        driver.launch(train_worker)
    except Exception as e:
        print(f"💀 Summ done messed up cuh {e}")
    
    finally:

        print("🔪 Initiating Termination Sequence...")

        # Ensure Sidecar finishes before termiantion
        time_count = 0
        while True:
            still_uploading = False
            for file in os.listdir("."):
                if file.startswith("UPLOAD_REQUEST_") and file.endswith(".json"):

                    if time_count % 60 == 0:
                        print(f"⏳ Sidecar is currently processing a final upload. Waiting for it to finish...")
                        print(f"Elapsed Time: {time_count//60} minute(s)")

                    still_uploading = True
                    break

            if not still_uploading:
                break

            time.sleep(5)
            time_count +=5
        
        # Terminate Sidecars
        uploader_process.terminate()
        uploader_process.join()      
        print("Shutdown complete.")

Writing parallel_hardware_trainer.py


In [4]:
%%writefile SpanMLMCollator.py
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dataclasses import dataclass
from transformers import AutoTokenizer
import json
import multiprocessing

class ConfigJson():
    def __init__(self, **kwargs):
        # Assign attributes from keyword arguments
        for key, value in kwargs.items():
            setattr(self, key, value)

    @classmethod
    def from_json(cls, json_path):
        with open(json_path, "r") as file:
            data = json.load(file)
        return cls(**data)



class SpanMLMCollator:

    # Define Init
    # Most of the things are just stored in the config lwk
    # Imma just pull from here
    def __init__(self, config = None, tokenizer = None, mlm_probability = None, mlm_use_span_masking = None, mlm_span_length = None):
        
        # Defaults
        self.mlm_probability = 0.15
        self.mlm_use_span_masking = False
        self.mlm_span_length = 3
        self.tokenizer = None

        # Override with Config (if provided)
        if (config is not None):
            if (isinstance(config, str)):
                try:
                    config = ConfigJson.from_json(config)
                except Exception as e:
                    raise ValueError(f"Blud was not a json. Either some sort of dictionary ahhh config or .json. Error: {e}")

            self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
            self.mlm_probability = config.mlm_probability
            self.mlm_use_span_masking = config.mlm_use_span_masking
            self.mlm_span_length = config.mlm_span_length

        if tokenizer is not None:
            self.tokenizer = tokenizer
        if mlm_probability is not None:
            self.mlm_probability = mlm_probability
        if mlm_use_span_masking is not None:
            self.mlm_use_span_masking = mlm_use_span_masking
        if mlm_span_length is not None:
            self.mlm_span_length = mlm_span_length
            
        if self.tokenizer is None:
            raise ValueError("Tokenizer is None. Either pass in the config or tokenizer")

        # AI HELP:
        # Create vocab for legal words to replace
        valid_ids = [i for i in range(len(self.tokenizer)) if i not in self.tokenizer.all_special_ids]
        self.valid_vocab = torch.tensor(valid_ids)

    # __call__() function returning a DataLoader with Span_masking
    def __call__(self, data):

        # Convert List of Dictionaries into 1 large tensor
        # inside, convert extract input_ids from dictionary
        batch_input_ids = [d["input_ids"] for d in data]
        input_ids = torch.tensor(batch_input_ids)
        labels = input_ids.clone()

        # Set mlm_prob
        mlm_prob = self.mlm_probability
        if (self.mlm_use_span_masking):
            mlm_prob /= self.mlm_span_length

        # Create the Masking Tensor
        masked_tensor = torch.full(input_ids.shape, mlm_prob, dtype=torch.float32)


        # HUH???


        # =====================================================================
        # THE GUARD STEP: Protect special tokens from being masked
        # =====================================================================
        # 1. Ask the tokenizer which tokens in each row are special (CLS, SEP, PAD)
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(seq, already_has_special_tokens=True) 
            for seq in input_ids.tolist()
        ]
        
        # 2. Convert that nested list into a PyTorch Boolean tensor
        special_tokens_map = torch.tensor(special_tokens_mask, dtype=torch.bool)
        
        # 3. Overwrite the probability in masked_tensor to 0.0 wherever special_tokens_map is True
        masked_tensor.masked_fill_(special_tokens_map, value=0.0)
        # =====================================================================


        # Apply Bernoulli to get 1s and 0s for the masks
        masked_tensor = torch.bernoulli(masked_tensor).bool()

        temp_clone = masked_tensor.clone()
        # Apply rolling for span masking
        if (self.mlm_use_span_masking):
            for i in range (1, self.mlm_span_length):
                rolled_tensor = torch.roll(temp_clone, shifts = i)
                rolled_tensor[:,:i] = False
                masked_tensor = masked_tensor | rolled_tensor

        # Mask all untampered tokens with -100
        labels[~masked_tensor] = -100
        
        # Create new Tensor w/ random values from 0-1
        type_tensor = torch.rand(input_ids.shape)

        # Create 80% mask_token_tensor
        mask_token_tensor = (type_tensor <= .8) & masked_tensor

        # Create 10% corrupted_token_tensor
        corrupted_token_tensor = (type_tensor > .8) & (type_tensor <= .9) & masked_tensor

        # Apply Mask tokens to input_ids
        input_ids[mask_token_tensor] = self.tokenizer.mask_token_id

        # LOOK HERE ##################################################
        
        # Apply corrupted tokens to input_ids
        num_to_replace = corrupted_token_tensor.sum().item()

        # Generate random indices for the valid_vocab_tensor
        indices = torch.randint(0,len(self.valid_vocab), (num_to_replace,))

        # Take the words form valid_vocab
        random_words = self.valid_vocab[indices]

        # Apply the words to the input_ids
        input_ids[corrupted_token_tensor] = random_words

        # ############################################################

        # Return Dictionary
        return {"input_ids": input_ids, "labels": labels}

Overwriting SpanMLMCollator.py


In [5]:
%%writefile model.py

##################################################
# Defines the HELM V1 architecture
# Inherited the PretrainedConfig and PreTrainedModel
# Utilizes many of the concepts found in Nvidia's 2024 nGPT architecture
# Initializes the Weights
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Hugging Face Core Method (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm"

    def __init__(
        self,
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 16,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,
        num_router_latents = 4,
        max_active_heads = 5,
        num_permanent_heads = 2,
        selection_threshold = 0.5,
        router_init_scale = 10.0,
        jitter_noise = 0.01,
        sparsity_lambda = 0.01, 
        router_aux_loss_coeff = 0.02,
        router_grad_clip = 0.05,    
        sparsity_warm_up_steps = 2000,
        use_sigmoid_scaling = True,
        ngpt_sqk_init_value = 1.0,  
        ngpt_sqk_init_scale = 0.03125,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,
        bias = False,
        use_ckpt = False,
        lr = 15e-4,
        use_exclusive_attention = False,
        **kwargs 
    ):  
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers 
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act 
        self.swiglu_s_init = swiglu_s_init
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length
        self.num_router_latents = num_router_latents
        self.max_active_heads = max_active_heads
        self.num_permanent_heads = num_permanent_heads
        self.num_elastic_heads = max_active_heads - num_permanent_heads
        self.selection_threshold = selection_threshold
        self.router_init_scale = router_init_scale
        self.jitter_noise = jitter_noise
        self.sparsity_lambda = sparsity_lambda
        self.router_aux_loss_coeff = router_aux_loss_coeff
        self.router_grad_clip = router_grad_clip     
        self.sparsity_warm_up_steps = sparsity_warm_up_steps
        self.use_sigmoid_scaling = use_sigmoid_scaling
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.bias = bias
        self.use_ckpt = use_ckpt
        self.lr = lr
        self.use_exclusive_attention = use_exclusive_attention

        super().__init__(**kwargs)


# Define Embedding Layer
class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size, 
            config.hidden_size, 
            padding_idx=config.pad_token_id
        )
        
    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):
        
        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings



# NOVEL: Multi-Latent Summary Router to decide which heads to use
class HELMMultiViewRouter(nn.Module):
   
    # Initialize the following:
    #   - Summary Query Matrix (q_down_proj)
    #   - Latent Importance Weights (l_i_weights)
    #   - Router_Init_Scale (tau)
    #   - Linear Router Gate (q_down_proj)
    def __init__(self, config):
        super().__init__()

        # Yoink some things from config
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_heads = self.config.num_elastic_heads

        # Summary Query Matrix size() : [hidden_size, num_router_latents]
        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias = config.bias
        )

        # Latent Importance Weights
        # size() [num_router_latents]
        self.l_i_weights = nn.Parameter(
            torch.ones(config.num_router_latents)
        )

        # Router_Init_Scale size() : [1]
        self.tau = nn.Parameter(torch.tensor(config.router_init_scale))

        # Linear Router Gate size() : [hidden_size, num_attention_heads - num_permanent_heads]
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            config.num_attention_heads - config.num_permanent_heads,
            bias = config.bias
        )

    # Pass in only Hidden States 
    # Don't pass in attention mask bc theres no attention here (duh)
    def forward(self, hidden_states, step_tensor):


        # Write vars for cleaner code
        q_down_proj = self.q_down_proj
        l_i_weights = self.l_i_weights
        tau = self.tau
        q_up_proj = self.q_up_proj 
        scale = self.scale
        num_elastic_heads = self.num_elastic_heads
        self.selection_threshold = self.config.selection_threshold


        
        # Norm Query Matrix
        # Requires .weight since the matrix was defined before
        q_down_proj = justnorm(q_down_proj.weight, dim = 1)

        # Multiply the hidden_state by Down projection (q_down_proj)
        # Call it "scanner"
        # Size: [b, s, hidden_size] * [hidden_size * num_router_latents] = [b, s, num_router_latents]
        scanner = F.linear(hidden_states, q_down_proj)

        # Apply Softmax to entire sequences (sequence level routing)
        scanner_softmax = F.softmax(scale * scanner, dim = 1)

        # Apply Transpose to allow for dimension matching
        # [b, s, num_router_latents] -> [b, num_router_latents, s]
        scanner_softmax = scanner_softmax.transpose(1,2)

        # Create Latent Vectors (Summary of the sequence in 4 vectors)
        # Size: [b,num_router_latents,s] * [b, s, hidden_size] = [b, num_router_latents, hidden_size]
        # Use bmm (batch matrix matric product) b/c [b, n_r_l, s] * [b, s, h_s] (dims don't match up normally)
        # Could've transposed, but this is more memory efficient
        latents = torch.bmm(scanner_softmax, hidden_states)

        # # Normalize the latents (or should we???)
        # latents = justnorm(latents)

        # Scale latents by Learnable important parameters (l_i_weights)
        # Softmax them first
        l_i_weights = F.softmax(l_i_weights, dim = 0)

        # Apply l_i_weights to latents
        # Sum the Latents together
        # Size: ([b, num_router_latents, hidden_size] * broadcast [1, num_router_latents, 1]) and sum the latents = [b, 1 (size of pooled_latents when we added them together), hidden_size]
        pooled_latents = (latents * l_i_weights.view(1, -1, 1)).sum(dim=1, keepdim = True)

        # # Normalize again (per nGPT requirements)
        # # Or should we? We need the weight of these latents so that sigmoid actually does its job
        # # Technically tau tries to help it, but maybe commenting this out makes more sense
        # pooled_latents = justnorm(pooled_latents)
        
        # Normalize q_up_proj      
        # Requires .weight since the matrix was defined before
        # size [num_permanent_heads - num_elastic_heads, hidden_size]
        q_up_proj = justnorm(q_up_proj.weight, dim = 1)

        # Multiply the latents by the classifer (q_up_proj)
        # Call it "class_scores"
        # Size: [b, 1, hidden_size] * [hidden_size, num_permanent_heads - num_elastic_heads] = [b, 1, num_permanent_heads - num_elastic_heads]
        class_scores = F.linear(pooled_latents, q_up_proj)

        # Multiply this by Tau (router_init_scale) and ngpt scaler sqrt(hidden_size), or should we???
        class_scores = class_scores * tau  # * scale

        # Sigmoid Scores
        # Size: still [b, 1, num_permanent_heads - num_elastic_heads], but with sigmoid scores
        sigmoid_scores = torch.sigmoid(class_scores)

        # Apply Router Gradient Clip (to prevent violent gradient updates if a specific head was wrong)
        if self.training and self.config.router_grad_clip > 0.0:
            
            def clip_router_gradients(raw_gradient):
                return torch.clamp(
                    raw_gradient,
                    min = -self.config.router_grad_clip,
                    max = self.config.router_grad_clip
            )

            sigmoid_scores.register_hook(clip_router_gradients)
            


        # NEW SELECTION STRATEGY: #########################
        # To prevent recompiling during top-k to threshold, we will move onto a slightly more refined approach:


        # Apply top_k logic:
        # indices dimension: [batch_size, 1, num_elastic_heads]
        _, indices = sigmoid_scores.topk(num_elastic_heads, dim = -1)
        # topk_mask size sum([b, 1, num_elastic_heads,num_permanent_heads - num_elastic_heads]) of the 3rd dimension = [b, 1 ,num_permanent_heads - num_elastic_heads]
        # Look at how one_hot works: https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html
        topk_mask = F.one_hot(indices, num_classes=sigmoid_scores.size(-1)).sum(dim=2).float()

        # Also calculate threshold mask:
        # indices dimension: [batch_size, 1, num_elastic_heads] (we just 0'd the sigmoid_scores values that are less then selection_threshold)
        threshold_mask = (sigmoid_scores > self.selection_threshold).float()

        # determine which phase of training this is in
        is_warmup = step_tensor < self.config.sparsity_warm_up_steps

        # Based on is_warmup, either use topk or threshold
        # flat_mask size [b, 1, num_permanent_heads - num_elastic_heads] (choose topk or threshold):
        # This is slightly bad for GPUs (since its a waste to calculation both), but at least the TPU doesn't have to recompile
        flat_mask = torch.where(is_warmup, topk_mask, threshold_mask)
    
        # # ###############################################



        # use_sigmoid_scaling = True: router_mask = Sigmoid values and 0s (Accuracy)
        # use_sigmoid_scaling = False: router_mask = 1s and 0s (Efficiency)
        if self.config.use_sigmoid_scaling:
            # Scale back to sigmoid scores
            flat_mask = flat_mask * sigmoid_scores


        # Apply STE for Dead Router Heads during backprop
        # Must happen after sigmoid_scaling or else torch could believe the sigmoid scaling are dynamically linked
        # .detach() Ignored during Backprop (Autograd doesn't see anything with.detach(), so when backprop happens, they disappear)
        # Forward pass: flat_mask- sigmoid_scores + sigmoid_scores = flat_mask
        # Backward pass: sigmoid_scores
        flat_mask = flat_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        # Add attention dimensions: [b, 1, num_permanent_heads - num_elastic_heads] -> [b, num_elastic_heads, 1, 1]
        router_mask = flat_mask.view(flat_mask.size(0), -1, 1, 1)

        # If permanent_heads are used, add the columns 
        # size(): [b, num_attention_heads, 1 , 1]
        if (self.config.num_permanent_heads > 0):
            permanent_head_scores = torch.ones(
                flat_mask.size(0), 
                self.config.num_permanent_heads, 
                1,
                1,
                device=router_mask.device, 
                dtype=router_mask.dtype
            )
            router_mask = torch.cat((permanent_head_scores, router_mask), dim = 1)

        # Calculate Training Losses Here:
        if self.training:
            # aux_loss: ensure that the router doesn't route the same head everytime (ignoring permanent heads)
            # Take each sequence in the batch, and average the sigmoid score within each head
            # squeeze = remove all dims = 1
            # size: [b, 1, num_permanent_heads - num_elastic_heads] -> [num_permanent_heads - num_elastic_heads]
            P = sigmoid_scores.mean(dim=0).squeeze()
            # Take each sequence in the batch, and average the sigmoid score only for the ones passing the threshold
            # If for example they all routed
            # size: [num_permanent_heads - num_elastic_heads]
            f = (sigmoid_scores > self.selection_threshold).float().mean(dim=0).squeeze() # Best approx of hard_mask here
            # Dot P*f and multiply by the number of elastic heads. This is the aux_loss
            raw_aux_loss = self.num_elastic_heads * torch.sum(f * P)
            self.aux_loss = self.config.router_aux_loss_coeff * raw_aux_loss
        
            # sparsity_loss: ensure that the model doesn't just turn on all the heads

            # Apply sparsity annealing for warm_up
            sparsity_warm_up_scale = torch.clamp(step_tensor / self.config.sparsity_warm_up_steps, max = 1.0)

            # Multiply set scaling factor (sparsity_lamdba) by the mean of the sigmoid_scores (remember sigmoid scores don't add up to 1)
            self.sparsity_loss = sparsity_warm_up_scale * self.config.sparsity_lambda * sigmoid_scores.mean()

        else:
            self.aux_loss = torch.tensor(0.0, device=hidden_states.device)
            self.sparsity_loss = torch.tensor(0.0, device=hidden_states.device)

        # Return Mask
        # [b, num_attention_heads, 1 , 1]
        return router_mask



# RoPE Class
class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())
    
    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings 
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.num_elastic_heads = config.num_elastic_heads
        self.d_head = config.hidden_size // config.num_attention_heads
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            config.hidden_size * 3,
            bias = config.bias 
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head, 
            config.max_position_embeddings, 
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.hidden_size))

        # Output Matrix
        self.output = nn.Linear(
            config.hidden_size,
            config.hidden_size,
            bias = config.bias
        )
    
    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = self.qkv(hidden_states)

        # Obtain Hidden Size
        batch_size, seq_len, hidden_size = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(hidden_size, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)

        # If Batch > 1 or model is in training mode: Cutting Losses and broadcast the mask
        if (batch_size > 1 or self.training):
            
            # Reshape q,k,v
            # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
            q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

            # Reshape q,k,v
            # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
            q = q.permute(0,2,1,3)
            k = k.permute(0,2,1,3)
            v = v.permute(0,2,1,3)

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor and justnorm to q and k (splice sqk too)
            q = sqk.to(q.dtype) * q 
            k = sqk.to(k.dtype) * k 

            # Apply Attention
            # Sclae by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v, 
                attn_mask=attention_mask,
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # # If broadcasting work, apply mask like this
            # # size(): [b, num_attention_heads, seq_len, d_head]
            # context_layer = context_layer * router_mask

            # if router_mask is not None:
            #     if router_mask.shape[1] != context_layer.shape[1]:
            #         # We need to map 'V' views to 'H' heads (e.g., 18 views -> 20 heads)
            #         # We treat the views like a 1D image and "stretch" it to the head count
            #         # [B, V, 1, 1] -> [B, 1, V]
            #         m = router_mask.squeeze(-1).squeeze(-1).unsqueeze(1)
            #         # Interpolate to [B, 1, H]
            #         m = F.interpolate(m, size=context_layer.shape[1], mode='nearest')
            #         # Reshape back to [B, H, 1, 1]
            #         router_mask = m.squeeze(1).unsqueeze(-1).unsqueeze(-1)

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape 
            # size(): [b, seq_len, num_attention_heads, d_head] 
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()
            
            # Flatten the last two dimensions: 
            # size(): [b, seq_len, num_hidden_size] 
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = self.output(context_reshaped)

        # Apply splicing logic and win on efficiency.
        else:

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # Reshape q,k,v
            # q, k, v size(): [b, seq_len, num_attention_heads, hidden_size]
            q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
            v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

            # Reshape q,k,v
            # q, k, v size(): [b, num_attention_heads, seq_len, hidden_size]
            q = q.permute(0,2,1,3)
            k = k.permute(0,2,1,3)
            v = v.permute(0,2,1,3)

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor and justnorm to q and k 
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced  
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced  

            # Flash Attention
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced, 
                attn_mask=attention_mask,
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values 
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()
            
            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims]

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=self.output.bias)
    
        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size, 
            2 * self.intermediate_size, 
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU 
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )
    
    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py
        
        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate 
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)
            
        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv = self.mlp_exp(hidden_states_opt1)

        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        
        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv = suv * uv  

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = self.mlp_proj(x_mlp)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate 
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp) 
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention (which defines RotaryEmbeddigs) + HELMMLP
# This is 1 transformer layer
class HELMBlock(nn.Module):

    # Define the Following:
    #   - HELMMultiViewRouter
    #   - HELMSelfAttention
    #   - HELMMLP
    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)
    
    # Define the forward pass
    # Extra: Return the aux_loss from the router
    def forward(self, hidden_states, attention_mask, step_tensor):
        router_mask = self.mlt_vw_rtr(hidden_states, step_tensor)
        aux_loss = self.mlt_vw_rtr.aux_loss
        sparsity_loss = self.mlt_vw_rtr.sparsity_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, aux_loss, sparsity_loss



# HELMModel - HELM without the head 
class HELMModel(nn.Module):

    # Define the following:
    #   - HELMEmbedding
    #   - HELMBlock
    def __init__(self, config):
        super().__init__()

        # Get the ckpt_attribute
        self.use_ckpt = config.use_ckpt

        # Embedding layer
        self.embedding = HELMEmbedding(config)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [HELMBlock(config) for _ in range(config.num_hidden_layers)]
        )
        
    
    # Forward Pass
    def forward(self, input_ids, attention_mask, current_step = None):


        # Reshape Attention Mask to be 4D for SDPA [batch_size, 1, 1, seq_len]
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).float()

        # Then make attend = 0 and mask = -inf to allow for Flash Attention compatitbility
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        # Convert the current_step to be infinity if null, or a tensor, or a tensor of the correct datatype if its already tensor
        # We did this because we don't want to pass in a non-tensor if we are using gradient_checkpointing
        if current_step is None:
            step_tensor = torch.tensor(float("inf"), device=input_ids.device)
        elif not isinstance(current_step, torch.Tensor):
            step_tensor = torch.tensor(current_step, device=input_ids.device)
        else:
            step_tensor = current_step

        # Pass input_ids through the input
        embeddings = self.embedding(input_ids)

        # Set Embeddings to be hidden_states
        hidden_states = embeddings 

        # Accumulate aux_loss and sparsity_loss
        total_aux_loss = 0
        total_sparsity_loss = 0

        # Run Tranformer Blocks
        for block in self.blocks:
            # Use Gradient Checkpointing
            if self.use_ckpt and self.training:
                hidden_states, aux_loss, sparsity_loss = torch.utils.checkpoint.checkpoint(
                    block,
                    hidden_states, 
                    attention_mask, 
                    step_tensor,
                    use_reentrant=False
                )
            # Or Standard Forward Pass
            else:
                hidden_states, aux_loss, sparsity_loss = block(hidden_states, attention_mask, step_tensor)
                
            total_aux_loss += aux_loss
            total_sparsity_loss += sparsity_loss

        # Return hidden state (feature extraction / context location prediction) & special losses
        # hidden_states: [b, seq_len, hidden_size]
        return hidden_states, total_aux_loss, total_sparsity_loss



# HELMModelforMaskedLM
class HELMForMaskedLM(PreTrainedModel):

    # Define the Config for the HF push_to_hub() function
    config_class = HELMConfig

    # Define the Following:
    #   - HELMModel
    #   - classifier
    #   - Head layer scaling vector
    def __init__(self, config):
        super().__init__(config)

        # Define from Config
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale

        # Define the Model
        self.model = HELMModel(config)

        # Define the head Layer
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias = config.bias
        )

        # Define the head layer scaling vetor 
        self.sz = nn.Parameter(torch.ones(config.vocab_size))

        # HF Function to call _init_weights() function
        self.post_init()

    # Initialize weights (pulled from ngpt model.py)
    def _init_weights(self, module):

        # If it's an nn.Linear, initialize it with the initializer_range
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        # If it's an nn.Linear, initialize it with the initializer_range also)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    
    # Forward pass
    def forward(self, input_ids, attention_mask, current_step = None):

        # Gather Context from the model
        # features: [b, seq_len, hidden_size]
        features, total_aux_loss, total_sparsity_loss = self.model(input_ids, attention_mask, current_step)

        # Scale / prepare sz
        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)

        # project features onto classifer
        # [b, seq_len, hidden_size] * [hidden_size, vocab_size] = [b, seq_len, vocab_size]
        unscaled_logits = self.classifier(features)

        # Scale the logits with sz
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits

        # Return Logits
        return logits, total_aux_loss, total_sparsity_loss

Writing model.py
